# Enhanced S3 to COG Converter with Chunked Processing

This notebook converts TIF files from S3 to Cloud Optimized GeoTIFFs (COGs) with:
- **Chunked processing** for memory-efficient handling of large files
- **Automatic AWS credential detection** (no .env file needed)
- **Download caching** to avoid re-downloading large files
- **COG validation** before uploading
- **Memory monitoring** and progress tracking

Author: Kyle Lesinger (Enhanced chunked version)

In [1]:
import os
import pandas as pd
import json
import tempfile
import boto3
import rasterio
from rasterio.windows import Window
from rasterio.enums import Resampling
from rasterio.warp import calculate_default_transform, reproject
from rasterio.io import MemoryFile
import rioxarray as rxr
import s3fs
import fsspec
from botocore.exceptions import NoCredentialsError, ClientError
from pathlib import Path
from datetime import datetime
import time
import numpy as np
import gc
import psutil
from tqdm import tqdm

print("✅ Libraries imported successfully!")
print(f"Boto3 version: {boto3.__version__}")
print(f"Rasterio version: {rasterio.__version__}")

✅ Libraries imported successfully!
Boto3 version: 1.37.3
Rasterio version: 1.4.3


In [2]:
# Add path for importing custom modules
import sys
from pathlib import Path

# Add the scripts directory to the Python path
scripts_dir = Path('../scripts').resolve()
if str(scripts_dir) not in sys.path:
    sys.path.insert(0, str(scripts_dir))

# Import functions from list_s3crawler_files module
from list_s3crawler_files import (
    load_drcs_data,
    get_tif_files_from_path,
    get_files_with_full_paths,
    list_available_directories
)

# Import COG and cache utilities
from cog_utilities import (
    check_cache_status,
    clear_cache,
    validate_cog,
    export_COG_PROFILE
)

# Import AWS S3 utilities
from aws_s3_utils import (
    initialize_s3_client,
    verify_s3_client,
    get_all_s3_keys
)

# Import batch processing utilities
from batch_processing import (
    process_file_batch,
    print_batch_summary
)

from memory_utils import (
    get_memory_usage,
    calculate_optimal_chunk_size,
    estimate_chunk_memory,
    format_bytes

)

from convert_utilities import (
    convert_to_proper_CRS_and_cogify_chunked
)
    
print("✅ Custom modules imported successfully!")
print(f"   Module path: {scripts_dir}")

✅ Memory monitoring utilities loaded
✅ Custom modules imported successfully!
   Module path: /home/jovyan/conversion_scripts/convert-files-and-move/scripts


# Useful links
<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">drcs_activations OLD Directory</a> -- You can view old directory file structure here.

<a href="https://docs.openveda.cloud/user-guide/content-curation/dataset-ingestion/file-preparation.html" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">VEDA docs for file naming conventions</a> -- Helps for understanding why/how we name content.

## List of new 2nd level directories

    "Sentinel-1"
    "Sentinel-2"
    "Landsat"
    "MODIS"
    "VIIRS"
    "ASTER"
    "MASTER"
    "ECOSTRESS"
    "Planet"
    "Maxar"
    "HLS"
    "IMERG"
    "GOES"
    "SMAP"
    "ICESat"
    "GEDI"
    "COMSAR"
    "UAVSAR"
    "WB-57"

In [3]:
# DO NOT CHANGE
DIR_OLD_BASE = 'drcs_activations'
DIR_NEW_BASE = 'drcs_activations_new'
BUCKET = 'nasa-disasters'

In [4]:

EVENT_NAME = '202309_Hurricane_Idalia'  #find the name within drcs_activations OLD Directory (see link above)
PRODUCT_NAME = 'landsat'      #find the name within drcs_activations OLD Directory (see link above)
PATH_OLD = f'{DIR_OLD_BASE}/{EVENT_NAME}/{PRODUCT_NAME}'  # Updated to use actual available directory

In [5]:
# Define COG profile for rasterio (DO NOT CHANGE)
COG_PROFILE = export_COG_PROFILE()

# Chunked processing configuration
CHUNK_CONFIG = {
    "default_chunk_size": 1024,  # Default chunk size in pixels
    "memory_limit_mb": 500,      # Memory limit per chunk in MB
    "show_progress": True,       # Show progress bars
    "enable_memory_monitoring": True  # Monitor memory usage
}

## Initialize AWS S3 Client with automatic credential detection

In [6]:
# Initialize AWS S3 Client using the imported function
s3_client, fs_read = initialize_s3_client(bucket_name=BUCKET, verbose=True)

# Verify S3 client is ready using the imported function
verify_s3_client(s3_client, bucket_name=BUCKET, verbose=True)

# Get all TIF files using the imported function
keys = get_all_s3_keys(s3_client, BUCKET, PATH_OLD, ".tif") if s3_client else []

if keys:
    print(f"✅ Found {len(keys)} .tif files in the S3 bucket.")
else:
    print("No keys found or S3 client not initialized")
    
keys

⚠️ S3 client initialized (limited bucket list access)
✅ Confirmed access to nasa-disasters bucket
✅ S3 filesystem (fsspec) initialized
✅ S3 client ready for operations
   Bucket: nasa-disasters
   Ready to process files
✅ Found 51 .tif files in the S3 bucket.


['drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230819/colorIR/LC08_colorInfrared_20230819_161332_019039.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230819/nat/LC08_naturalColor_20230819_161332_019039.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230819/true/LC08_trueColor_20230819_161332_019039.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/colorIR/LC09_colorInfrared_20230820_160648_018038.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/colorIR/LC09_colorInfrared_20230820_160712_018039.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/colorIR/LC09_colorInfrared_20230820_160735_018040.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/nat/LC09_naturalColor_20230820_160648_018038.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/nat/LC09_naturalColor_20230820_160712_018039.tif',
 'drcs_activat

## Configure bucket and paths (no need to create session manually)

In [7]:
def return_bucket_info(config):
    """
    Extract bucket information from configuration and return as dictionary.
    
    Args:
        config: Configuration dictionary containing bucket and prefix information
    
    Returns:
        Dictionary with bucket and prefix information
    """
    # Configure bucket and paths (no need to create session manually)
    bucket_name = config["cog_data_bucket"]
    raw_data_bucket = config["raw_data_bucket"]
    raw_data_prefix = config["raw_data_prefix"]
    
    cog_data_bucket = config['cog_data_bucket']
    cog_data_prefix = config["cog_data_prefix"]
    
    print(f"Configuration loaded:")
    print(f"  Source bucket: {raw_data_bucket}")
    print(f"  Source prefix: {raw_data_prefix}")
    print(f"  Target bucket: {cog_data_bucket}")
    print(f"  Target prefix: {cog_data_prefix}")

    return {
        "bucket_name": bucket_name,
        "raw_data_bucket": raw_data_bucket,
        "raw_data_prefix": raw_data_prefix,
        "cog_data_bucket": cog_data_bucket,
        "cog_data_prefix": cog_data_prefix
    }

## Define Chunked COG Conversion Function

This function handles the conversion of files to Cloud Optimized GeoTIFFs with:
- Chunked processing to handle large files
- Memory monitoring
- Progress tracking
- Proper CRS and caching

In [8]:
# Check current cache status using the imported function
check_cache_status()

📊 Cache Status:
  - Directory: data_download/
  - Total files: 325
  - Total size: 70.52 GB

📁 Cached files (first 10):
  - drcs_activations/20230719_SevereWx_NC/aria/ARIA_DPM_Sentinel-1_North_Carolina_Tornado.tif (4.2 MB)
  - drcs_activations/20230719_SevereWx_NC/aria/ARIA_DPMraw_Sentinel-1_North_Carolina_Tornado.tif (4.2 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC08_L1TP_20230706_015035_colorInfrared.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC08_L1TP_20230706_015035_naturalColor.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC08_L1TP_20230706_015035_trueColor.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC09_L1TP_20230628_015035_colorInfrared.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC09_L1TP_20230628_015035_naturalColor.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/landsat/LC09_L1TP_20230628_015035_trueColor.tif (178.9 MB)
  - drcs_activations/20230719_SevereWx_NC/sentinel1

(325, 75720310728)

In [9]:
import re

def simple_process_files(keys, filter_str, rename_func, target_dir, EVENT_NAME):
    """
    Simple wrapper to process files with minimal code.
    
    Args:
        keys: List of all S3 keys
        filter_str: Can be:
            - String to filter files (e.g. 'S1_WTR')
            - Regex pattern object (e.g. re.compile(r'.*S2A.*mosaic'))
            - Callable function that returns True/False
        rename_func: Your custom rename function
        target_dir: Target directory (e.g. "Sentinel-1/opera_dswx")
        EVENT_NAME: Event name
    
    Returns:
        Processing results DataFrame
    """
    # 1. Filter files based on type of filter_str
    if callable(filter_str):
        # If it's a function
        filtered_files = [i for i in keys if filter_str(i)]
    elif hasattr(filter_str, 'search'):
        # If it's a compiled regex pattern
        filtered_files = [i for i in keys if filter_str.search(i)]
    elif isinstance(filter_str, str) and filter_str.startswith('r"') or filter_str.startswith("r'"):
        # If it's a regex string (e.g., r'pattern')
        pattern = re.compile(filter_str[2:-1])  # Remove r" or r'
        filtered_files = [i for i in keys if pattern.search(i)]
    else:
        # Default: simple string contains
        filtered_files = [i for i in keys if filter_str in i]
    
    # 2. Test renaming
    print(f"Testing filenames:")
    for f in filtered_files:
        print(f"  {rename_func(f, EVENT_NAME)}")
    
    # 3. Setup config
    config = {
        "data_acquisition_method": "s3",
        "raw_data_bucket": BUCKET,
        "raw_data_prefix": PATH_OLD,
        "cog_data_bucket": BUCKET,
        "cog_data_prefix": f'{DIR_NEW_BASE}/{target_dir}',
        "local_output_dir": f"output/{EVENT_NAME}",
        "transformation": {}
    }
    return_bucket_info(config)
    
    # 4. Process files
    print("\n" + "="*50)
    print("🌊 Processing Files (Chunked)")
    print("="*50)
    
    def chunked_converter(name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, local_output_dir=None):
        return convert_to_proper_CRS_and_cogify_chunked(
            name, BUCKET, cog_filename, cog_data_bucket, cog_data_prefix, s3_client, COG_PROFILE,
            local_output_dir, chunk_config=CHUNK_CONFIG
        )

    results = process_file_batch(
        file_list=filtered_files,
        s3_client=s3_client,
        config=config,
        filename_creator_func=rename_func,
        processing_func=chunked_converter,
        event_name=EVENT_NAME,
        save_metadata=True,
        save_csv=True,
        verbose=True,
        BUCKET=BUCKET
    )
    
    print_batch_summary(results)
    return results

# Process files

In [10]:
keys

['drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230819/colorIR/LC08_colorInfrared_20230819_161332_019039.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230819/nat/LC08_naturalColor_20230819_161332_019039.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230819/true/LC08_trueColor_20230819_161332_019039.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/colorIR/LC09_colorInfrared_20230820_160648_018038.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/colorIR/LC09_colorInfrared_20230820_160712_018039.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/colorIR/LC09_colorInfrared_20230820_160735_018040.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/nat/LC09_naturalColor_20230820_160648_018038.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/nat/LC09_naturalColor_20230820_160712_018039.tif',
 'drcs_activat

# Landsat 8, naturalColor

In [11]:
def create_cog_filename_landsat_pre_event(f, EVENT_NAME):
    """Extract date from Landsat filename and move to end with formatted date."""
    filename_stem = Path(f).stem
    
    # Parse Landsat filename pattern: LC0X_colorType_YYYYMMDD_time_pathrow
    landsat_pattern = re.match(r'^(LC0[89])_(\w+)_(\d{8})_(.*)$', filename_stem)
    
    if landsat_pattern:
        satellite = landsat_pattern.group(1)
        color_type = landsat_pattern.group(2)
        date_str = landsat_pattern.group(3)
        path_row_info = landsat_pattern.group(4)
        
        # Format date as YYYY-MM-DD
        formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
        
        # Create new filename: EVENT_NAME_satellite_colorType_pathrow_date_day
        cog_filename = f'{EVENT_NAME}_pre_event_{satellite}_{color_type}_{path_row_info}_{formatted_date}_day.tif'
    else:
        # Fallback to original logic
        date_match = re.search(r'(20\d{6})', filename_stem)
        if date_match:
            date_str = date_match.group(1)
            formatted_date = f"{date_str[:4]}-{date_str[4:6]}-{date_str[6:8]}"
            filename_parts = filename_stem.replace(date_str + '_', '')
            cog_filename = f'{EVENT_NAME}_{filename_parts}_{formatted_date}_day.tif'
        else:
            cog_filename = f'{EVENT_NAME}_{filename_stem}_day.tif'
    
    return cog_filename

# Define filename creator functions for different file types
pattern = re.compile(r'LC08.*naturalColor.*\.tif$')

# Test functions
print("Testing WM filename:")
filter_ =  [f for f in keys if pattern.search(f)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_landsat_pre_event(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_161332_019039_2023-08-19_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_160023_017037_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_160047_017038_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_160111_017039_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_160135_017040_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_160159_017041_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_154738_015036_2023-08-23_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_15482_015037_2023-08-23_day.tif


In [12]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_landsat_pre_event, 
                                target_dir = "Landsat/naturalColor", 
                                EVENT_NAME = EVENT_NAME)


Testing filenames:
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_161332_019039_2023-08-19_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_160023_017037_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_160047_017038_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_160111_017039_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_160135_017040_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_160159_017041_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_154738_015036_2023-08-23_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_naturalColor_15482_015037_2023-08-23_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202309_Hurricane_Idalia/landsat
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Landsat/naturalColor

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202309

Band 1:  86%|████████▌ | 62/72 [00:02<00:00, 22.64chunks/s]


   [MEMORY] High usage: 580.7 MB, forcing cleanup...

   [MEMORY] High usage: 584.8 MB, forcing cleanup...


   [BAND 2/3] Processing...


Band 2:   0%|          | 0/72 [00:00<?, ?chunks/s]

...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmppkqwjdq8_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp41ar8i52.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/202309_Hurricane_Idalia_pre_event_LC08_naturalColor_15482_015037_2023-08-23_day.tif
   [MEMORY] Final: 960.3 MB (Change: +5.0 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202309_Hurricane_Idalia_pre_event_LC08_naturalColor_15482_015037_2023-08-23_day.tif

✅ Batch processing complete: 8 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/files_converted.csv
📁 COGs saved locally to: output/202309_Hurricane_Idalia

📊 BATCH PROCESSING SUMMARY
Total files processed: 8
Successful: 8
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09

# Landsat 8, colorInfrared

In [13]:
# Define filename creator functions for different file types
pattern = re.compile(r'LC08.*colorInfrared.*\.tif$')

# Test functions
print("Testing WM filename:")
filter_ =  [f for f in keys if pattern.search(f)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_landsat_pre_event(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")




Testing WM filename:
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_161332_019039_2023-08-19_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_160023_017037_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_160047_017038_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_160111_017039_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_160135_017040_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_160159_017041_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_154738_015036_2023-08-23_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_15482_015037_2023-08-23_day.tif


In [14]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_landsat_pre_event, 
                                target_dir = "Landsat/colorInfrared", 
                                EVENT_NAME = EVENT_NAME)

Testing filenames:
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_161332_019039_2023-08-19_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_160023_017037_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_160047_017038_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_160111_017039_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_160135_017040_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_160159_017041_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_154738_015036_2023-08-23_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_15482_015037_2023-08-23_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202309_Hurricane_Idalia/landsat
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Landsat/colorInfrared

🌊 Processing Files (Chunked)
✅ Local output directory ready: outp

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmppud8v6iz_temp.tif



...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmp42_16xql_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmps9c1iftx.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/colorInfrared/202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_15482_015037_2023-08-23_day.tif
   [MEMORY] Final: 1042.6 MB (Change: +58.3 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202309_Hurricane_Idalia_pre_event_LC08_colorInfrared_15482_015037_2023-08-23_day.tif

✅ Batch processing complete: 8 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Landsat/colorInfrared/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Landsat/colorInfrared/files_converted.csv
📁 COGs saved locally to: output/202309_Hurricane_Idalia

📊 BATCH PROCESSING SUMMARY
Total files processed: 8
Successful: 8
Failed: 0
Success rate: 100.0%
Timestamp: 

# Landsat 8, trueColor

In [15]:
# Define filename creator functions for different file types
pattern = re.compile(r'LC08.*trueColor.*\.tif$')

# Test functions
print("Testing WM filename:")
filter_ =  [f for f in keys if pattern.search(f)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_landsat_pre_event(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")


Testing WM filename:
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_161332_019039_2023-08-19_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_160023_017037_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_160047_017038_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_160111_017039_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_160135_017040_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_160159_017041_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_154738_015036_2023-08-23_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_15482_015037_2023-08-23_day.tif


In [16]:
# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_landsat_pre_event, 
                                target_dir = "Landsat/trueColor", 
                                EVENT_NAME = EVENT_NAME)

Testing filenames:
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_161332_019039_2023-08-19_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_160023_017037_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_160047_017038_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_160111_017039_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_160135_017040_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_160159_017041_2023-08-21_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_154738_015036_2023-08-23_day.tif
  202309_Hurricane_Idalia_pre_event_LC08_trueColor_15482_015037_2023-08-23_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202309_Hurricane_Idalia/landsat
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Landsat/trueColor

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202309_Hurricane_Idalia

[1/8] Pr

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=4, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=4, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=4, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpihfk1elh_temp.tif



...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=4, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=4, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=4, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmppbgwvfdh_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmpjcoykxm1.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/202309_Hurricane_Idalia_pre_event_LC08_trueColor_15482_015037_2023-08-23_day.tif
   [MEMORY] Final: 990.0 MB (Change: -58.5 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202309_Hurricane_Idalia_pre_event_LC08_trueColor_15482_015037_2023-08-23_day.tif

✅ Batch processing complete: 8 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Landsat/trueColor/files_converted.csv
📁 COGs saved locally to: output/202309_Hurricane_Idalia

📊 BATCH PROCESSING SUMMARY
Total files processed: 8
Successful: 8
Failed: 0
Success rate: 100.0%
Timestamp: 2025-09-09T02:00:34.8

In [17]:
keys

['drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230819/colorIR/LC08_colorInfrared_20230819_161332_019039.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230819/nat/LC08_naturalColor_20230819_161332_019039.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230819/true/LC08_trueColor_20230819_161332_019039.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/colorIR/LC09_colorInfrared_20230820_160648_018038.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/colorIR/LC09_colorInfrared_20230820_160712_018039.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/colorIR/LC09_colorInfrared_20230820_160735_018040.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/nat/LC09_naturalColor_20230820_160648_018038.tif',
 'drcs_activations/202309_Hurricane_Idalia/landsat/pre_event/20230820/nat/LC09_naturalColor_20230820_160712_018039.tif',
 'drcs_activat

# Landsat 9, naturalColor

In [18]:

pattern = re.compile(r'LC09.*naturalColor.*\.tif$')

# Test functions
print("Testing WM filename:")
filter_ =  [f for f in keys if pattern.search(f)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_landsat_pre_event(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_160648_018038_2023-08-20_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_160712_018039_2023-08-20_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_160735_018040_2023-08-20_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_155428_016038_2023-08-22_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_15544_016037_2023-08-22_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_155452_016039_2023-08-22_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_155516_016040_2023-08-22_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_155540_016041_2023-08-22_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_15564_016042_2023-08-22_day.tif


In [19]:
# Define filename creator functions for different file types

# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_landsat_pre_event, 
                                target_dir = "Landsat/naturalColor", 
                                EVENT_NAME = EVENT_NAME)

Testing filenames:
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_160648_018038_2023-08-20_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_160712_018039_2023-08-20_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_160735_018040_2023-08-20_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_155428_016038_2023-08-22_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_15544_016037_2023-08-22_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_155452_016039_2023-08-22_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_155516_016040_2023-08-22_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_155540_016041_2023-08-22_day.tif
  202309_Hurricane_Idalia_pre_event_LC09_naturalColor_15564_016042_2023-08-22_day.tif
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202309_Hurricane_Idalia/landsat
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Landsat/na

   [BAND 2/3] Processing...


   [BAND 3/3] Processing...


   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmpzlfvy65a_temp.tif



...

   [VERIFY] Checking reprojected data...
   [VERIFY] RGB file without nodata - all pixel values are valid
   [VERIFY] Band 1: min=14, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 2: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [VERIFY] Band 3: min=1, max=255, center sample non-zero=1000000/1000000
            Estimated data coverage: 100.0% (from distributed samples)
   [COGIFY] Creating COG from reprojected file...
   [NODATA] Data type: uint8
   [NODATA] Using nodata value 0 for uint8 data
   [PREDICTOR] Data type: uint8, using PREDICTOR=2
   [WRITE] Writing temporary GeoTIFF with chunked processing...


Reading input: /tmp/tmp30ksmf95_temp.tif



   [CONVERT] Converting GeoTIFF to COG...
   [CONVERT] Using rio-cogeo for conversion...


Adding overviews...
Updating dataset tags...
Writing output to: /tmp/tmp3byb_7gp.tif


   [VALIDATE] Checking COG validity...
   [VALIDATE] ⚠️ COG validation warnings
      - Compression 'zstd' may not be optimal for COG
   [UPLOAD] Uploading to S3...
   [SUCCESS] ✅ Uploaded to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/202309_Hurricane_Idalia_pre_event_LC09_naturalColor_15564_016042_2023-08-22_day.tif
   [MEMORY] Final: 990.3 MB (Change: -52.9 MB)
✅ Chunked COG conversion function defined with memory-efficient processing
   ✅ Generated and saved COG: 202309_Hurricane_Idalia_pre_event_LC09_naturalColor_15564_016042_2023-08-22_day.tif

✅ Batch processing complete: 9 files processed
📊 Uploaded metadata to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/metadata.json
📝 Saved processing log to s3://nasa-disasters/drcs_activations_new/Landsat/naturalColor/files_converted.csv
📁 COGs saved locally to: output/202309_Hurricane_Idalia

📊 BATCH PROCESSING SUMMARY
Total files processed: 9
Successful: 9
Failed: 0
Success rate: 100.0%
Timestamp: 2025-0

# Landsat 9, colorInfrared

In [20]:

pattern = re.compile(r'LC09.*colorInfrared\.tif$')

# Test functions
print("Testing WM filename:")
filter_ =  [f for f in keys if pattern.search(f)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_landsat_pre_event(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:


In [21]:
# Define filename creator functions for different file types

# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_landsat_pre_event, 
                                target_dir = "Landsat/naturalColor", 
                                EVENT_NAME = EVENT_NAME)

Testing filenames:
Configuration loaded:
  Source bucket: nasa-disasters
  Source prefix: drcs_activations/202309_Hurricane_Idalia/landsat
  Target bucket: nasa-disasters
  Target prefix: drcs_activations_new/Landsat/naturalColor

🌊 Processing Files (Chunked)
✅ Local output directory ready: output/202309_Hurricane_Idalia

✅ Batch processing complete: 0 files processed
📁 COGs saved locally to: output/202309_Hurricane_Idalia

📊 BATCH PROCESSING SUMMARY
Total files processed: 0
Successful: 0
Failed: 0
Success rate: N/A
Timestamp: 2025-09-09T02:05:22.540882


# Landsat 9, trueColor

In [22]:

pattern = re.compile(r'LC09.*trueColor.*\.tif$')

# Test functions
print("Testing WM filename:")
filter_ =  [f for f in keys if pattern.search(f)]

for idx,i in enumerate(filter_):
    test_wm = create_cog_filename_landsat(filter_[idx], EVENT_NAME)
    print(f"  {test_wm}")

Testing WM filename:


NameError: name 'create_cog_filename_landsat' is not defined

In [ ]:
# Define filename creator functions for different file types

# Process S1 WTR files
results1 = simple_process_files(keys=keys, 
                                filter_str = pattern, 
                                rename_func = create_cog_filename_landsat_pre_event, 
                                target_dir = "Landsat/trueColor", 
                                EVENT_NAME = EVENT_NAME)

## Check STATUS of file conversion and upload

<a href="https://data.disasters.openveda.cloud/browseui/browseui/#drcs_activations_new/" target="_blank" rel="noopener noreferrer" style="color: blue; font-size: 20px;">Disasters Bucket</a> -- You can view that the files actually made it to their correct destination.

## Memory Usage Summary

You can check the final memory usage and cleanup

In [ ]:
# Final memory cleanup and report
gc.collect()
final_memory = get_memory_usage()
print(f"\n📊 Memory Usage Summary:")
print(f"  Current memory usage: {final_memory:.1f} MB")
print(f"  Available memory: {psutil.virtual_memory().available / 1024 / 1024:.1f} MB")
print(f"  Memory percent used: {psutil.virtual_memory().percent:.1f}%")